In [13]:
import numpy as np
import matplotlib.pyplot as plt

from torchvision.datasets import CIFAR10
from torchvision import transforms

from torch.utils.data import DataLoader, random_split, Subset

from tqdm.auto import tqdm

from vit import *
from eval import *

In [17]:


device = "cuda" if torch.cuda.is_available() else "cpu"

transform = transforms.Compose([
    transforms.ToTensor(),
    transforms.Normalize(
        mean=(0.4914, 0.4822, 0.4465),
        std=(0.2470, 0.2435, 0.2616)
    )
])


train_dataset = CIFAR10(root="data", train=True, transform=transform, download=True)
test_dataset = CIFAR10(root="data", train=False, transform=transform, download=True)

torch.manual_seed(42)

val_ratio = 0.2
# num_val_samples = int(val_ratio * len(train_dataset))
# num_train_samples = len(train_dataset) - num_val_samples

train_dataset = Subset(train_dataset, range(0, 500))
num_val_samples = 100
num_train_samples = 400

train_dataset, val_dataset = random_split(train_dataset, [num_train_samples, num_val_samples])

batch_size = 128

train_dataloader = DataLoader(
    train_dataset,
    batch_size = batch_size,
    shuffle = True
)

val_dataloader = DataLoader(
    val_dataset,
    batch_size = batch_size,
    shuffle = False
)

test_dataloader = DataLoader(
    test_dataset,
    batch_size = batch_size,
    shuffle = False
)

config = ViTConfig()
model = ViT(config).to(device)

epochs = 500
lr = 3e-4
weight_decay = 1e-4

optimizer = torch.optim.AdamW(params=model.parameters(), lr=lr, weight_decay=weight_decay)

scheduler = torch.optim.lr_scheduler.CosineAnnealingLR(
    optimizer,
    T_max=epochs
)
loss_fn = torch.nn.CrossEntropyLoss()

for epoch in range(epochs):
    train_loss = 0.0
    train_correct_preds = 0

    pbar = tqdm(total=num_train_samples, unit="samples", desc = f"Epoch: {epoch+1}")

    model.train()
    for i, train_batch in enumerate(train_dataloader):
        x_train, y_train = train_batch
        x_train, y_train = x_train.to(device), y_train.to(device)

        optimizer.zero_grad()

        logits = model(x_train)

        loss = loss_fn(logits, y_train)
        train_loss += loss.item()

        loss.backward()
        optimizer.step()

        if (epoch % 5 == 0):
            train_correct_preds += correct_preds(logits, y_train)

        pbar.update(len(x_train))

    pbar.close()
    scheduler.step()

    val_correct_preds = 0
    val_loss = 0.0


    if (epoch % 5 == 0):
        model.eval()
        for i, val_batch in enumerate(val_dataloader):
            x_val, y_val = val_batch
            x_val, y_val = x_val.to(device), y_val.to(device)

            logits = model(x_val)

            loss = loss_fn(logits, y_val)
            val_loss += loss.item()
            
            val_correct_preds += correct_preds(logits, y_val)

        print(f"Epoch: {epoch+1} | "
              f"train_loss: {train_loss / len(train_dataloader) :.4f} | "
              f"train_acc: {100 * train_correct_preds / num_train_samples:.2f} | "
              f"val_loss: {val_loss / len(val_dataloader) :.4f} | "
              f"val_acc: {100 * val_correct_preds / num_val_samples :.2f}\n")

# LEARN: LR schedulers, pytorch broadcasting

Epoch: 1: 100%|██████████| 400/400 [00:00<00:00, 7166.63samples/s]


Epoch: 1 | train_loss: 2.3783 | train_acc: 8.25 | val_loss: 2.2905 | val_acc: 13.00



Epoch: 6: 100%|██████████| 400/400 [00:00<00:00, 7926.46samples/s]


Epoch: 6 | train_loss: 2.2278 | train_acc: 20.75 | val_loss: 2.2269 | val_acc: 24.00



Epoch: 11: 100%|██████████| 400/400 [00:00<00:00, 8703.55samples/s]


...

Epoch: 491: 100%|██████████| 400/400 [00:00<00:00, 7945.68samples/s]


Epoch: 491 | train_loss: 0.0262 | train_acc: 100.00 | val_loss: 4.4574 | val_acc: 32.00



Epoch: 496: 100%|██████████| 400/400 [00:00<00:00, 10094.35samples/s]


Epoch: 496 | train_loss: 0.0277 | train_acc: 100.00 | val_loss: 4.4574 | val_acc: 32.00



Epoch: 500: 100%|██████████| 400/400 [00:00<00:00, 8986.38samples/s]
